# Capítulo 7 · QAOA — Quantum Approximate Optimization Algorithm

## Objetivos

1. Comprender QAOA como algoritmo variacional para optimización combinatoria.
2. Implementar QAOA para el problema MaxCUT en un grafo pequeño.
3. Analizar la calidad de la solución en función del número de capas $p$.

---

## 7B.1 El problema MaxCUT

Dado un grafo $G=(V,E)$, MaxCUT busca una bipartición $(S, \bar{S})$ de los vértices que maximice el número de aristas entre $S$ y $\bar{S}$.

El Hamiltoniano de coste es:

$$H_C = \sum_{(i,j) \in E} w_{ij} \frac{I - Z_i Z_j}{2}$$

y el ansatz QAOA de profundidad $p$ es:

$$|\boldsymbol{\gamma}, \boldsymbol{\beta}\rangle = e^{-i\beta_p H_B} e^{-i\gamma_p H_C} \cdots e^{-i\beta_1 H_B} e^{-i\gamma_1 H_C} |s\rangle$$

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_aer import AerSimulator
from src.visualization import QuantumVisualization

print('Módulos cargados.')

In [ ]:
# Grafo de 4 nodos (ejemplo pedagógico)
n_nodes = 4
edges = [(0, 1, 1.0), (0, 2, 1.0), (1, 3, 1.0), (2, 3, 1.0)]  # (u, v, peso)

print('Grafo MaxCUT:')
for u, v, w in edges:
    print(f'  Arista ({u},{v}) peso={w}')

# Hamiltoniano de coste
def maxcut_hamiltonian(n: int, edges: list) -> SparsePauliOp:
    """Construye H_C para MaxCUT."""
    ops = []
    for u, v, w in edges:
        # (I - Z_u Z_v) / 2  ×  weight
        I_term = 'I' * n
        ZZ_term = list('I' * n)
        ZZ_term[n - 1 - u] = 'Z'
        ZZ_term[n - 1 - v] = 'Z'
        ops.append((''.join(I_term), w / 2))
        ops.append((''.join(ZZ_term), -w / 2))
    return SparsePauliOp.from_list(ops).simplify()

H_cost = maxcut_hamiltonian(n_nodes, edges)
print('\nHamiltoniano de coste:')
print(H_cost)

In [ ]:
def qaoa_circuit(n: int, edges: list, gamma: list, beta: list) -> QuantumCircuit:
    """Construye el circuito QAOA de profundidad p.

    Parámetros
    ----------
    n : int
        Número de qubits (= número de nodos).
    edges : list
        Lista de aristas (u, v, w).
    gamma : list
        Ángulos del operador de coste (longitud p).
    beta : list
        Ángulos del operador de mezcla (longitud p).
    """
    p = len(gamma)
    qc = QuantumCircuit(n)

    # Estado inicial: superposición uniforme
    qc.h(range(n))

    for layer in range(p):
        # Operador de problema (ZZ)
        for u, v, w in edges:
            qc.cx(u, v)
            qc.rz(2 * gamma[layer] * w, v)
            qc.cx(u, v)

        # Operador de mezcla (X)
        for i in range(n):
            qc.rx(2 * beta[layer], i)

    return qc


def qaoa_cost(params: np.ndarray, n: int, edges: list, p: int, H_matrix) -> float:
    """Evalúa el valor esperado de H_C para los parámetros dados."""
    gamma = params[:p]
    beta  = params[p:]
    qc = qaoa_circuit(n, edges, gamma, beta)
    sv = Statevector(qc)
    return float(-np.real(sv.data.conj() @ H_matrix @ sv.data))  # negativo para min


# Optimización para p=1
p = 1
H_matrix = H_cost.to_matrix()

energy_history = []

def tracked_cost(params):
    val = qaoa_cost(params, n_nodes, edges, p, H_matrix)
    energy_history.append(-val)
    return val

np.random.seed(7)
params0 = np.random.uniform(0, np.pi, 2 * p)

result = minimize(tracked_cost, params0, method='COBYLA',
                  options={'maxiter': 300})

best_cut = -result.fun
print(f'QAOA p={p}:')
print(f'  Corte aproximado = {best_cut:.4f}')
print(f'  Corte máximo exacto = {len(edges)}')
print(f'  Ratio de aproximación = {best_cut / len(edges):.4f}')

In [ ]:
# Visualización
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(energy_history, color='#58a6ff', linewidth=1.5)
ax.axhline(len(edges), color='#f78166', linestyle='--',
           label=f'Óptimo clásico = {len(edges)}')
ax.set_xlabel('Iteración')
ax.set_ylabel('Valor del corte ⟨H_C⟩')
ax.set_title(f'Convergencia QAOA p={p} — MaxCUT 4 nodos')
ax.legend()
ax.grid(alpha=0.3)
ax.set_facecolor('#161b22')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

## 7B.2 Ejercicios propuestos

1. Repite QAOA para $p = 2$ y $p = 3$. ¿Mejora la calidad de la solución con $p$?

2. Aplica QAOA al problema de 3-coloración de un grafo expresado como un Hamiltoniano de penalización.

3. Implementa el Balanaced Partition Problem (partición equilibrada) y resuélvelo con QAOA.